<a href="https://colab.research.google.com/github/Pradeep-Kumar-Panga/DecodingComplexities/blob/main/OneHotEncoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One Hot Encoding
The rigght way vs. the common production pitfall

Link for detailed explanation: [One Hot Encoding](https://youtu.be/ffcg0uV4w5Y)
Companion Blog Post: [One Hot Encosing](https://www.pradeeppanga.com/2026/08/one-hot-encoding.html)

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

## Setup
Create a toy train and test sets, intentionally with a different set of categories in each - to simulate the real world test/production data mismatches

In [ ]:
X_train = pd.DataFrame({
    "city": ["Bangalore", "Mumbai", "Delhi", "Bangalore", "Mumbai"]
})

X_test = pd.DataFrame({
    "city": ["Bangalore", "Mumbai"]
})

y_train = [1, 1, 0, 1, 0]

print("Train:", X_train["city"].unique(), "; Shape:", X_train.shape, "\n")
print("Test:", X_test["city"].unique(), "; Shape:", X_test.shape, "\n")

Train: ['Bangalore' 'Mumbai' 'Delhi'] ; Shape: (5, 1) 

Test: ['Bangalore' 'Mumbai'] ; Shape: (2, 1) 



## The Pitfall
pandas.get_dummies() called separately on train and test sets

This results in a failure when test set does not contain same categories as the train set.

In [ ]:
X_train_dummies = pd.get_dummies(X_train)
X_test_dummies = pd.get_dummies(X_test)

print("Train dummies:\n", X_train_dummies)
print("Shape after Encoding:", X_train_dummies.shape, "\n")

print("Test dummies:\n", X_test_dummies)
print("Shape after Encoding:", X_test_dummies.shape, "\n")


Train dummies:
    city_Bangalore  city_Delhi  city_Mumbai
0            True       False        False
1           False       False         True
2           False        True        False
3            True       False        False
4           False       False         True
Shape after Encoding: (5, 3) 

Test dummies:
    city_Bangalore  city_Mumbai
0            True        False
1           False         True
Shape after Encoding: (2, 2) 



In [ ]:
# Fit the model using the toy dataset created
model = LogisticRegression().fit(X_train_dummies, y_train)

try:
    model.predict(X_test_dummies)
except ValueError as e:
    print("ValueError:", e)


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- city_Delhi



## The Fix
Scikit-learn's OneHotEncoder, fit once on training data, then re-use on test data

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore")

train_encoded = encoder.fit_transform(X_train)
test_encoded = encoder.transform(X_test)

train_encoded_df = pd.DataFrame(train_encoded.toarray(), columns=encoder.get_feature_names_out())
test_encoded_df = pd.DataFrame(test_encoded.toarray(), columns=encoder.get_feature_names_out())

print("Train Enoded:\n", train_encoded_df)
print("Train Encoded shape:", train_encoded.shape, "; Columns:", list(encoder.get_feature_names_out()))

print("Test Encoded:\n", test_encoded_df)
print("Test Encoded shape:", test_encoded.shape,  "; Columns:", list(encoder.get_feature_names_out()))

Train Enoded:
    city_Bangalore  city_Delhi  city_Mumbai
0             1.0         0.0          0.0
1             0.0         0.0          1.0
2             0.0         1.0          0.0
3             1.0         0.0          0.0
4             0.0         0.0          1.0
Train Encoded shape: (5, 3) ; Columns: ['city_Bangalore', 'city_Delhi', 'city_Mumbai']
Test Encoded:
    city_Bangalore  city_Delhi  city_Mumbai
0             1.0         0.0          0.0
1             0.0         0.0          1.0
Test Encoded shape: (2, 3) ; Columns: ['city_Bangalore', 'city_Delhi', 'city_Mumbai']


### Not just switching to Scikit-Learn
We used fit_transform for train, but only transform for test, Why?
fit_transform overrides what encoder has already learned.

In [ ]:
incorrect_encoder = OneHotEncoder(handle_unknown="ignore")

incorrect_train_encoded = incorrect_encoder.fit_transform(X_train)
incorrect_train_encoded_df = pd.DataFrame(incorrect_train_encoded.toarray(), columns=incorrect_encoder.get_feature_names_out())

incorrect_test_encoded = incorrect_encoder.fit_transform(X_test)
incorrect_test_encoded_df = pd.DataFrame(incorrect_test_encoded.toarray(), columns=incorrect_encoder.get_feature_names_out())

print("Train Enoded:\n", incorrect_train_encoded_df)
print("Train Encoded shape:", incorrect_train_encoded.shape, "; Columns:", list(incorrect_encoder.get_feature_names_out()))

print("Test Encoded:\n", incorrect_test_encoded_df)
print("Test Encoded shape:", incorrect_test_encoded.shape,  "; Columns:", list(incorrect_encoder.get_feature_names_out()))

Train Enoded:
    city_Bangalore  city_Delhi  city_Mumbai
0             1.0         0.0          0.0
1             0.0         0.0          1.0
2             0.0         1.0          0.0
3             1.0         0.0          0.0
4             0.0         0.0          1.0
Train Encoded shape: (5, 3) ; Columns: ['city_Bangalore', 'city_Mumbai']
Test Encoded:
    city_Bangalore  city_Mumbai
0             1.0          0.0
1             0.0          1.0
Test Encoded shape: (2, 2) ; Columns: ['city_Bangalore', 'city_Mumbai']


### Why handle_unknown?
If a new category shows up, handle_unknown = "ignore" tells encoder to encode it as all zeroes.

In [ ]:
new_data = pd.DataFrame({
    "city": ["Chennai"]
})

new_dummies = encoder.transform(new_data)
new_dummies_df = pd.DataFrame(new_dummies.toarray(), columns=encoder.get_feature_names_out())

print("New dummies:\n", new_dummies_df)
print("Test Encoded shape:", new_dummies.shape,  "; Columns:", list(encoder.get_feature_names_out()))

New dummies:
    city_Bangalore  city_Delhi  city_Mumbai
0             0.0         0.0          0.0
Test Encoded shape: (1, 3) ; Columns: ['city_Bangalore', 'city_Delhi', 'city_Mumbai']
